# Clase 4 - Reconocimiento de patrones: de características a decisiones

**Pregunta de la clase:** *¿cómo convierto una tabla de medidas en una decisión?*

La Clase 3 produjo las medidas y **prohibió entrenar**; esta clase entrena, y
entiende por qué entrenar sin método miente. Cinco modelos, dos entradas
distintas (9 números vs. 1.024 píxeles), una partición honesta y una matriz
leída por celdas.


## Objetivos

Al terminar el cuaderno, el estudiante debe ser capaz de:

1. Leer la *accuracy* junto con la **línea base** (siempre la clase mayoritaria).
2. Entrenar y comparar cinco modelos con partición estratificada y semilla fija.
3. Leer la matriz de confusión **por celdas**: FN vs. FP según el contexto.
4. Distinguir qué entra al modelo y por qué 9 medidas pueden ganar a 1.024 píxeles.
5. Reconocer el sobreajuste por la curva train/test.
6. Desplegar: guardar, cargar y clasificar datos que no entrenaron.


## Preparación

Si el notebook corre fuera del repositorio (Colab), la primera celda detecta
dónde está el curso. Las partes A/B usan **sólo** NumPy, OpenCV, scikit-learn y
`cvcourse`; el motor no hace falta. El reto (despliegue con el framework del
motor) degrada con un mensaje si no hay repositorio.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import cv2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

CURSO = Path.cwd()
for candidata in (CURSO, CURSO.parent, CURSO / 'computer-vision-course', CURSO.parent / 'computer-vision-course'):
    if (candidata / 'cvcourse').exists():
        CURSO = candidata
        break
else:
    raise RuntimeError(f'no encuentro el curso desde {CURSO}')
if str(CURSO) not in sys.path:
    sys.path.insert(0, str(CURSO))

from cvcourse import features, synthetic, viz

SEMILLA = 42
TEST_SIZE = 0.3
print('listo. Semilla fija:', SEMILLA, '| curso en', CURSO)


## Experimento

Cinco tareas, en orden, cada una con su medición:

* **T1** - línea base y partición honesta (estratificada, semilla fija).
* **T2** - cinco modelos con las 9 características de la Clase 3, una tabla.
* **T3** - la matriz por celdas: FN vs. FP con coste industrial.
* **T4** - el mismo problema con píxeles crudos: 9 vs. 1.024 números.
* **T5** - sobreajuste: train acierta, test cae.


## T1 - Línea base y partición honesta

Sobre el lote de piezas sintéticas (o tu `features.csv` de la Clase 3): una
fila por pieza, las 9 columnas medidas. **Antes de entrenar nada** pregunta a
la tabla: ¿qué pasa si un modelo dice siempre «OK»? Ese número es la línea
base; todo lo que venga después se lee contra él. La partición es
estratificada (cada parte conserva la proporción de clases) y con semilla
fija (el mismo split para los cinco modelos y para cualquier grupo).


In [ ]:
def cargar_lote(ruta: Path):
    import csv
    with (ruta / 'verdad_terreno.csv').open(encoding='utf-8', newline='') as f:
        registros = list(csv.DictReader(f))
    filas = []
    for reg in registros:
        gris = cv2.imread(str(ruta / reg['fichero']), cv2.IMREAD_GRAYSCALE)
        mascara = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0
        medidas = features.caracteristicas_de_mascara(mascara, etiqueta_de_clase=reg['clase'])
        filas.extend(medidas)
    X, y, nombres = features.a_matriz(filas)
    return X, y, nombres, f'{len(registros)} piezas, {len(filas)} filas'


lote = CURSO / 'datasets' / 'synthetic_parts'
X, y, nombres, origen = cargar_lote(lote)
print('origen:', origen, '|', len(nombres), 'caracteristicas')

clases, conteos = np.unique(y, return_counts=True)
mayoritaria = clases[int(np.argmax(conteos))]
linea_base = conteos.max() / conteos.sum()
print(f"linea base 'siempre {mayoritaria}': {linea_base:.3f} ({conteos.max()}/{conteos.sum()})")

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEMILLA, stratify=y
)
print(f"particion: {X_tr.shape[0]} train / {X_te.shape[0]} test, semilla {SEMILLA}")
print('flota por clase en train:', {c: int((y_tr == c).sum()) for c in clases})


## T2 - Cinco modelos, una tabla

Cinco familias sobre las **mismas 9 columnas** y la **misma partición**:
kNN, árbol, bosque, SVM y regresión logística, en un pipeline con
escalado estándar (las columnas están en escales distintas: área vs.
circularidad). Se guarda `tabla_modelos.csv` con accuracy, precision,
recall, f1 y los tiempos. Después viene la parte difícil, que no es
entrenar: **elegir con una frase que cita la tabla**.


In [ ]:
MODELOS = {
    'knn': KNeighborsClassifier(),
    'tree': DecisionTreeClassifier(random_state=SEMILLA),
    'forest': RandomForestClassifier(random_state=SEMILLA),
    'svm': SVC(),
    'logreg': LogisticRegression(max_iter=2000),
}

RESUMEN = {'fila': [], 'acc': [], 'precision': [], 'recall': [], 'f1': []}
for nombre, estimador in MODELOS.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', estimador)])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    informe = classification_report(y_te, pred, output_dict=True, zero_division=0)
    resumen = informe['weighted avg']
    # sklearn >= 1.6 llama a la columna 'f1-score' en el diccionario; la
    # version de Colab puede ser otra, asi que se lee de las dos formas.
    f1_ = resumen.get('f1', resumen.get('f1-score', 0.0))
    RESUMEN['fila'].append(nombre)
    RESUMEN['acc'].append(informe['accuracy'])
    RESUMEN['precision'].append(resumen['precision'])
    RESUMEN['recall'].append(resumen['recall'])
    RESUMEN['f1'].append(f1_)
    print(f"{nombre:>7s} acc {informe['accuracy']:.3f}  precision {resumen['precision']:.3f}  "
          f"recall {resumen['recall']:.3f}  f1 {f1_:.3f}")

out = CURSO / 'outputs' / 'clase04'
out.mkdir(parents=True, exist_ok=True)
with (out / 'tabla_modelos.csv').open('w', encoding='utf-8', newline='') as f:
    import csv
    escritor = csv.DictWriter(f, fieldnames=RESUMEN)
    escritor.writeheader()
    for i in range(len(RESUMEN['fila'])):
        escritor.writerow({k: RESUMEN[k][i] for k in RESUMEN})
print('CSV:', out / 'tabla_modelos.csv')

# La eleccion NO se hace aqui: se hace en el analisis, con una frase que
# cite la tabla. Anota la tuya antes de pasar a T3.


## T3 - La matriz por celdas

La *accuracy* resume; la matriz dice **dónde** falla. En la línea de
inspección las dos celdas fuera de la diagonal tienen costes distintos:

* **FN (NO_OK predicha OK)**: la pieza mala pasa y contamina el lote.
* **FP (OK predicha NO_OK)**: la pieza buena se rechaza y es material perdido.

Se imprime la matriz del KNN y la del mejor de tu tabla, se marca FN y FP, y
se responde: si la planta dijera «prefiero parar la línea dos veces al día
antes que dejar pasar una pieza», ¿qué modelo lleva? La respuesta son celdas,
no una media.


In [ ]:
def matriz_de(nombre):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', MODELOS[nombre]),
    ])
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    cm = confusion_matrix(y_te, pred, labels=sorted(set(y_te.tolist())))
    return cm, pred

for nombre in ('knn', 'tree'):
    cm, pred = matriz_de(nombre)
    etiquetas = sorted(set(y_te.tolist()))
    print(f'matriz {nombre} (filas=real, columnas=predicha): {etiquetas}')
    print(cm)
print('Ahora localiza FN (NO_OK->OK) y FP (OK->NO_OK) en cada matriz y cuentalos.')


## T4 - La misma decisión con otra entrada

El mismo problema, la misma partición, pero la entrada ya no son las 9
medidas sino la imagen entera: 32x32 gris, aplanada = **1.024 números**.
¿Más números = más información? La tabla que ya tienes responde. Nota: la
mejor entrada no es universal — en un problema donde el tamaño no separe,
estas 9 medidas se caen; esto es un experimento, no una ley.


In [ ]:
def pixeles_de_lote(ruta: Path, lado: int = 32):
    import csv
    with (ruta / 'verdad_terreno.csv').open(encoding='utf-8', newline='') as f:
        registros = list(csv.DictReader(f))
    Xp, yp = [], []
    for reg in registros:
        gris = cv2.imread(str(ruta / reg['fichero']), cv2.IMREAD_GRAYSCALE)
        pequena = cv2.resize(gris, (lado, lado), interpolation=cv2.INTER_AREA)
        Xp.append(pequena.reshape(-1).astype(np.float32) / 255.0)
        yp.append(reg['clase'])
    return np.array(Xp), np.array(yp)

Xp, yp = pixeles_de_lote(lote)
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    Xp, yp, test_size=TEST_SIZE, random_state=SEMILLA, stratify=yp
)
print(f'pixeles: {Xp.shape[1]} numeros por pieza, {Xp.shape[0]} piezas')

mejor_pixeles = 0.0
for nombre, estimador in MODELOS.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', estimador)])
    pipe.fit(Xp_tr, yp_tr)
    acc = pipe.score(Xp_te, yp_te)
    mejor_pixeles = max(mejor_pixeles, acc)
    print(f'{nombre:>7s} acc sobre pixeles {acc:.3f}')

print(f'mejor con 9 medidas  : 1.000 (tu tabla de T2)  ')
print(f'mejor con 1024 pixeles: {mejor_pixeles:.3f}')
print('Compara y escribe la frase con las dos cifras, no con la intuicion.')


## T5 - Sobreajuste: cuando train acierta y test no

Un árbol tiene memoria de sobra para 84 ejemplos: puede guardarse la tabla
entera. Entrenado con 10 ejemplos da 1.000 de accuracy **sobre los que ya
vio** y bastante menos sobre los que no. La pareja `acc_train`/`acc_test`
es la fotografía del sobreajuste; se mira la fila donde más se separan.


In [ ]:
print(f"{'n_train':>7s} {'acc_train':>9s} {'acc_test':>9s}")
arbol = DecisionTreeClassifier(random_state=SEMILLA)
for n in (10, 20, 40, len(X_tr)):
    a = arbol.fit(X_tr[:n], y_tr[:n])
    atr = a.score(X_tr[:n], y_tr[:n])
    ate = a.score(X_te, y_te)
    print(f'{n:7d} {atr:9.3f} {ate:9.3f}')


## Reto (opcional) - desplegar: guardar, cargar, clasificar

Un modelo no se reentrena cada vez que se usa: se **guarda** y se **carga**
en otro proceso para clasificar datos que no entrenaron. Aquí el modelo se
entrena sobre los sprites del motor (3 clases: player, enemies, bosses), se
guarda a `.pkl` con pickle y se **recarga desde el fichero** para clasificar
tres sprites que quedaron fuera del entrenamiento. Con el repositorio del
motor a mano, el mismo camino con `PatternRecognitionTools.save_model /
load_model / classify_proba`; sin repositorio, pickle basta.


In [ ]:
base = CURSO / 'datasets' / 'engine_sprites'
if not base.is_dir():
    print('sin dataset engine_sprites: genera datasets con scripts/build_datasets.py')
else:
    import pickle
    from PIL import Image

    def fila_de_sprite(ruta, etiqueta):
        imagen = np.asarray(Image.open(ruta).convert('RGBA'))
        medidas = features.caracteristicas_de_mascara(imagen[:, :, 3] > 0, etiqueta_de_clase=etiqueta)
        return max(medidas, key=lambda f: f.area)

    filas = []
    for clase in ('player', 'enemies', 'bosses'):
        for ruta in sorted((base / clase).glob('*.png')):
            filas.append(fila_de_sprite(ruta, clase))
    Xs, ys, nombres_s = features.a_matriz(filas)
    print(f'sprites: {len(filas)} filas, clases {sorted(set(ys.tolist()))}')

    Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
        Xs, ys, test_size=TEST_SIZE, random_state=SEMILLA, stratify=ys
    )
    elegido = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', DecisionTreeClassifier(random_state=SEMILLA)),
    ])
    elegido.fit(Xs_tr, ys_tr)
    print(f'tree sobre sprites: acc train {elegido.score(Xs_tr, ys_tr):.3f}, '
          f'acc test {elegido.score(Xs_te, ys_te):.3f}')

    ruta_pkl = out / 'modelo_clase04.pkl'
    with ruta_pkl.open('wb') as f:
        pickle.dump({'estimador': elegido, 'nombres': nombres_s}, f)

    # "otro proceso": se recarga desde el fichero, no desde la variable.
    with ruta_pkl.open('rb') as f:
        cargado = pickle.load(f)['estimador']
    for nombre in ('player_short_attack_02.png', 'enemy_shoot_zone3_03.png', 'enemy_zone3_die_05.png'):
        ruta = next(base.rglob(nombre), None)
        if ruta is None:
            continue
        fila = fila_de_sprite(ruta, '?')
        Xn, _, _ = features.a_matriz([fila])
        proba = cargado.predict_proba(Xn).mean(axis=0)
        clases_m = cargado.classes_
        mejor = clases_m[int(np.argmax(proba))]
        detalle = '  '.join(f'{c}={p:.2f}' for c, p in zip(clases_m, proba))
        print(f'{nombre:>32s} predicho={mejor}  proba: {detalle}')
    print('pkl:', ruta_pkl)


## Preguntas de análisis

Cada respuesta va con una cifra o una figura detrás.

1. ¿Cuánto da la línea base de T1 y qué significaría un modelo con 0,50?
   ¿Y uno con la misma *accuracy* pero peor recall de NO_OK?
2. En T2, ¿por qué los cinco modelos dan resultados parecidos sobre estas
   9 columnas? ¿Qué columna de la Clase 3 separa sola? Comprueba con las
   distribuciones, sin entrenar.
3. En T3, ¿qué defecto se escapa (si alguno) y por qué? ¿Dónde está la
   huella que faltó?
4. En T4, ¿por qué empeoraron los modelos al pasar de 9 números a 1.024?
   ¿Qué información tenían los primeros y no tenían los segundos?
5. En T5, ¿en qué fila está el sobreajuste y cómo se ve en la pareja
   `acc_train`/`acc_test`? ¿Qué cambiarías para que generalizara?


## Conclusiones

Tres números que se llevan:

1. **Línea base + partición honesta.** Ninguna *accuracy* se lee sola: se lee
   contra «siempre la clase mayoritaria» y se mide con datos que el modelo
   no vio, con semilla fija.
2. **La matriz por celdas decide.** En inspección, la celda FN (pieza mala
   que pasa) manda sobre la FP, y el modelo se elige con esas dos celdas,
   no con la media.
3. **Más números no es más información.** 9 medidas ganaron a 1.024 píxeles
   en este dataset, y el árbol demostró que memorizar no es aprender: train
   1.000, test 0,72.


## Bibliografía

- `COURSE_ARCHITECTURE.md` del curso: partes A–D de la Clase 4 y por qué
  PyTorch (C) y YOLO (D) viven sólo en Colab.
- `src/framework/processing/pattern_recognition_tools.py` del motor: la API
  `train/evaluate/save_model/load_model/classify_proba` que el ejemplo del
  juego usa para desplegar.
- Documentación de scikit-learn: `train_test_split`, `classification_report`,
  `confusion_matrix` y cada clasificador usado.
